# Task-6: Play Store App Category Analysis
This notebook details the data cleaning, filtering, and analysis of Google Play Store apps to compare the average installs and revenue of **Free vs. Paid** apps within the **top 3 app categories**.

### Filter Criteria:
1. **Installs**: $\ge 10,000$ installs
2. **Revenue**: $\ge \$10,000$ for paid apps (free apps are kept)
3. **Android Version**: $> 4.0$ (Strictly greater than 4.0, excluding 4.0 itself)
4. **Size**: $> 15	ext{M}$ (Megabytes)
5. **Content Rating**: `'Everyone'`
6. **App Name**: $\le 30$ characters (including spaces and special characters)


In [ ]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the play store data
df = pd.read_csv('Dataset/play_store.csv')
print(f"Original shape: {df.shape}")


In [ ]:
# 1. Drop corrupt row 10472
if 10472 in df.index and df.loc[10472, 'Category'] == '1.9':
    df = df.drop(10472)
    print("Dropped corrupt row 10472.")

# 2. Clean Installs
df['Installs_numeric'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)

# 3. Clean Price
df['Price_numeric'] = df['Price'].str.replace('$', '', regex=False).astype(float)

# 4. Calculate Revenue
df['Revenue'] = df['Installs_numeric'] * df['Price_numeric']

# 5. Clean Size (Convert to Megabytes)
def clean_size(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val == 'Varies with device':
        return np.nan
    if val.endswith('M') or val.endswith('m'):
        return float(val[:-1])
    elif val.endswith('k') or val.endswith('K'):
        return float(val[:-1]) / 1024.0
    return np.nan

df['Size_MB'] = df['Size'].apply(clean_size)

# 6. Clean Android Version (strictly > 4.0)
def is_more_than_4_0(ver_str):
    if not isinstance(ver_str, str):
        return False
    if ver_str == 'Varies with device':
        return False
    match = re.match(r'^(\d+(?:\.\d+)*)', ver_str)
    if not match:
        return False
    parts = [int(p) for p in match.group(1).split('.')]
    while len(parts) < 2:
        parts.append(0)
    return tuple(parts) > (4, 0)

df['Android_Ver_more_than_4_0'] = df['Android Ver'].apply(is_more_than_4_0)

# 7. App Name Length
df['App_Name_Len'] = df['App'].str.len()

# Deduplicate to ensure each unique app name is analyzed only once
df['Reviews_numeric'] = pd.to_numeric(df['Reviews'], errors='coerce')
df = df.sort_values('Reviews_numeric', ascending=False).drop_duplicates(subset=['App'], keep='first')

print("Columns cleaned and prepared.")


In [ ]:
# Filter definitions
f_installs = df['Installs_numeric'] >= 10000
f_android = df['Android_Ver_more_than_4_0']
f_size = df['Size_MB'] > 15
f_content = df['Content Rating'] == 'Everyone'
f_name = df['App_Name_Len'] <= 30

# Conditional revenue filter: keep all free apps, but exclude paid apps with revenue < 10,000
f_revenue = (df['Type'] == 'Free') | ((df['Type'] == 'Paid') & (df['Revenue'] >= 10000))

# Combine all filters
filtered_df = df[f_installs & f_android & f_size & f_content & f_name & f_revenue].copy()

print(f"Filtered dataset shape: {filtered_df.shape}")
print(f"Free apps matching: {len(filtered_df[filtered_df['Type'] == 'Free'])}")
print(f"Paid apps matching: {len(filtered_df[filtered_df['Type'] == 'Paid'])}")


In [ ]:
# Determine the top 3 categories by total installs
top_3_categories = filtered_df.groupby('Category')['Installs_numeric'].sum().sort_values(ascending=False).head(3).index.tolist()
print(f"Top 3 categories by total installs: {top_3_categories}")

# Filter the data for only these top 3 categories
analysis_df = filtered_df[filtered_df['Category'].isin(top_3_categories)].copy()


In [ ]:
# Compute average installs and average revenue for free vs. paid apps within these categories
agg_df = analysis_df.groupby(['Category', 'Type']).agg(
    Avg_Installs=('Installs_numeric', 'mean'),
    Avg_Revenue=('Revenue', 'mean'),
    App_Count=('App', 'count')
).reset_index()

print("Aggregated metrics for the Top 3 Categories:")
print(agg_df)


In [ ]:
# Prepare data for plotting
# Ensure both Free and Paid exist for each category in the dataframe (fill with 0 if missing)
categories = top_3_categories
plot_data = []
for cat in categories:
    for t in ['Free', 'Paid']:
        row = agg_df[(agg_df['Category'] == cat) & (agg_df['Type'] == t)]
        if not row.empty:
            plot_data.append({
                'Category': cat,
                'Type': t,
                'Avg_Installs': row['Avg_Installs'].values[0],
                'Avg_Revenue': row['Avg_Revenue'].values[0]
            })
        else:
            plot_data.append({
                'Category': cat,
                'Type': t,
                'Avg_Installs': 0.0,
                'Avg_Revenue': 0.0
            })

plot_df = pd.DataFrame(plot_data)

# Create the figure
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Add Bars for Avg Installs
free_installs = plot_df[plot_df['Type'] == 'Free']
paid_installs = plot_df[plot_df['Type'] == 'Paid']

# Helper to format numbers for text labels on top of bars
def format_bar_label(val):
    if val >= 1e6:
        return f"{val/1e6:.1f}M"
    elif val >= 1e3:
        return f"{val/1e3:.1f}K"
    elif val > 0:
        return f"{val:.0f}"
    return "0"

free_text = free_installs['Avg_Installs'].apply(format_bar_label)
paid_text = paid_installs['Avg_Installs'].apply(format_bar_label)

fig.add_trace(
    go.Bar(
        name='Free Avg Installs',
        x=free_installs['Category'],
        y=free_installs['Avg_Installs'],
        text=free_text,
        textposition='outside',
        textfont=dict(color='#3B82F6', size=10),
        marker_color='#3B82F6', # Premium vibrant blue
        opacity=0.85,
        offsetgroup=1
    ),
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        name='Paid Avg Installs',
        x=paid_installs['Category'],
        y=paid_installs['Avg_Installs'],
        text=paid_text,
        textposition='outside',
        textfont=dict(color='#60A5FA', size=10),
        marker_color='#60A5FA', # Premium lighter blue
        opacity=0.85,
        offsetgroup=2
    ),
    secondary_y=False
)

# 2. Add Line/Markers for Avg Revenue
# Group by category and compute avg revenue for paid apps specifically to overlay
paid_revenue = plot_df[plot_df['Type'] == 'Paid']

fig.add_trace(
    go.Scatter(
        name='Paid Avg Revenue ($)',
        x=paid_revenue['Category'],
        y=paid_revenue['Avg_Revenue'],
        mode='lines+markers',
        line=dict(color='#F59E0B', width=4), # Premium amber/gold line
        marker=dict(size=10, symbol='diamond', color='#D97706'),
    ),
    secondary_y=True
)

# Update layout
fig.update_layout(
    title=dict(
        text='Average Installs vs. Average Revenue (Free vs. Paid) for Top 3 Categories',
        font=dict(size=18, family='Outfit, sans-serif')
    ),
    xaxis=dict(
        title='App Category',
        tickfont=dict(size=12, family='Inter, sans-serif')
    ),
    yaxis=dict(
        title=dict(text='Average Installs', font=dict(color='#1E3A8A')),
        tickfont=dict(color='#1E3A8A', size=11),
        gridcolor='rgba(229, 231, 235, 0.5)'
    ),
    yaxis2=dict(
        title=dict(text='Average Revenue ($)', font=dict(color='#D97706')),
        tickfont=dict(color='#D97706', size=11),
        overlaying='y',
        side='right',
        showgrid=False
    ),
    barmode='group',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=550
)

# Show figure
fig.show()

# Save standalone graph
fig.write_image('Screenshots/Graph1.png', scale=2)
print("Graph1.png saved successfully in Screenshots directory.")


### Insights and Observations:
1. **Category Dominance**: The top 3 categories by total installs are analyzed. Depending on the exact subset, these typically represent categories like `GAME` or `FAMILY`.
2. **Installs vs. Revenue**: Free apps generally secure a significantly higher average number of installs compared to Paid apps. However, Paid apps in these top categories generate substantial average revenue per app, presenting a trade-off between user acquisition volume and direct monetization.
3. **Monetization Viability**: While Free apps rely heavily on ads or in-app purchases (not fully captured by the download-price product), the direct revenue of Paid apps (exceeding $10k per app on average after filters) highlights a lucrative path for niche utility or high-quality gaming apps.
